In [ ]:
import pandas as pd
from evo.data_converters.common.objects.downhole_collection import DownholeCollection, ColumnMapping, HoleCollars

input_collars_df = pd.DataFrame(
    {
        "hole_index": [1, 2, 3],
        "hole_id": ["BH001", "BH002", "BH003"],
        "x": [174.76, 174.77, 174.78],
        "y": [-36.85, -36.86, -36.87],
        "z": [10.5, 12.3, 9.8],
        "final_depth": [20.0, 20.0, 20.0],
        "date": ["2024-01-15", "2024-01-16", "2024-01-17"],
    }
)

input_measurements_df = pd.DataFrame(
    {
        "hole_index": [1, 1, 1, 2, 2],
        "hole_id": ["BH001", "BH001", "BH001", "BH002", "BH002"],
        "SCPT_DPTH": [0.5, 1.0, 1.5, 0.5, 1.0],
        "SCPT_RES": [2.5, 3.2, 4.1, 1.8, 2.9],
        "SCPT_FRES": [1.2, 1.5, 1.3, 1.1, 1.4],
    }
)

collars = HoleCollars(df=input_collars_df)
dhc = DownholeCollection(
    collars=collars,
    measurements=[input_measurements_df],
    column_mapping=ColumnMapping(DEPTH_COLUMNS=["SCPT_DPTH"]),
    name="test",
)

In [ ]:
from python_ags4 import AGS4
from evo.data_converters.common.objects.downhole_collection.tables import DistanceTable

collars_df = dhc.collars.df

# get the first depth table... we'll probably want to loop over all tables and look for ones with SCPT column names.
measurements_df = dhc.get_measurement_tables(filter=[DistanceTable])[0].df

# Create LOCA table from HoleCollars
loca_table = pd.DataFrame(
    {
        "LOCA_ID": collars_df["hole_id"],
        "LOCA_NATE": collars_df["x"],
        "LOCA_NATN": collars_df["y"],
        "LOCA_GL": collars_df["z"],
        "LOCA_STAR": collars_df["date"],
    }
)

# SCPT table from measurements
scpt_table = pd.DataFrame(
    {
        "LOCA_ID": measurements_df["hole_id"],
        "SCPT_DPTH": measurements_df["SCPT_DPTH"],
        "SCPT_RES": measurements_df["SCPT_RES"],
        "SCPT_FRES": measurements_df["SCPT_FRES"],
    }
)

# Create the tables and headings dictionaries
tables = {"LOCA": loca_table, "SCPT": scpt_table}

headings = {"LOCA": loca_table.columns.tolist(), "SCPT": scpt_table.columns.tolist()}

# Write to AGS file
AGS4.dataframe_to_AGS4(tables, headings, "output.ags")